# Wallet leakage

Issue #6. How much card spend leaves YAPEAL for other financial providers, to whom, how it developed 2021–2023, and whether the growth is more customers or more leakage per customer. Charts go to `figures/leakage/`, the chart functions live in `src/plots.py` so the dashboard can reuse them.

Definitions: leakage = spend in the categories `cash` and `savings`, whose counterparts are financial providers (assumption A1). Conservative variant: only the counterparts that are 100 % e-commerce, i.e. card top-ups (Revolut, Binance, Coinbase, Twint). The bank rows (PostFinance, UBS, Kantonalbank) are point of sale and could be ATM withdrawals.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import mappings as M
from src import plots, style
from src.data import load_customer_table, load_sow, load_sow_counterpart, monthly_active
from src.features import build_customer_features

style.apply()
pd.set_option("display.width", 200)

sow = load_sow()
cp = load_sow_counterpart()
feats = build_customer_features(load_customer_table())

monthly = sow.groupby("date").agg(spend=("total_amount", "sum")).join(monthly_active(sow))
monthly["spend_per_active"] = monthly["spend"] / monthly["active_customers"]
quarterly_spend = sow.groupby("quarter")["total_amount"].sum()
fin = cp[cp["category"].isin(["cash", "savings"])].copy()
print(sow.shape, cp.shape, fin["top_counterpart"].unique())

## the base: customers and spend

In [ ]:
ax = plots.active_customers(monthly)
plots.save_fig(ax.figure, "01_active_customers", "leakage")
print(f"active customers: {monthly['active_customers'].iloc[0]:.0f} (Jan 21) → {monthly['active_customers'].max():.0f} (peak) → {monthly['active_customers'].iloc[-1]:.0f} (Dec 23)")
print(f"spend per active customer per month: median {monthly['spend_per_active'].median():.0f} CHF")

In [ ]:
cat_share = feats[[c for c in feats if c.startswith("cat_") and not c.endswith("_share")]].sum()
cat_share = (cat_share / cat_share.sum() * 100).rename(lambda c: c.replace("cat_", ""))
cat_share = cat_share[cat_share > 0.5]
ax = plots.category_shares(cat_share)
plots.save_fig(ax.figure, "02_category_shares", "leakage")
cat_share.sort_values(ascending=False).round(1).head(8)

## cash is not cash

the counterparts of `cash` and `savings`, and the check that the named ones are card top-ups: e-commerce share and amount per transaction.

In [ ]:
ax = plots.financial_counterparts(fin)
plots.save_fig(ax.figure, "03_financial_counterparts", "leakage")

check = fin.groupby("top_counterpart").apply(lambda g: pd.Series({
    "amount_kchf": g["total_amount"].sum() / 1000,
    "share": g["total_amount"].sum() / fin["total_amount"].sum(),
    "ecom": (g["ecom_perc"] * g["total_amount"]).sum() / g["total_amount"].sum(),
    "chf_per_tx": g["total_amount"].sum() / g["n_transactions"].sum(),
}), include_groups=False).sort_values("amount_kchf", ascending=False)
check.round(2)

Revolut, Binance, Coinbase and Twint are 100 % e-commerce, that is a card funding an app. PostFinance, UBS and Kantonalbank are point of sale with 140–250 CHF per transaction. `other` in `cash` is three quarters point of sale. Both readings are carried below: all of it (A1) and online only.

## leakage over time

In [ ]:
fin["group"] = fin["top_counterpart"].map(M.LEAKAGE_GROUPS).fillna("other")
by_q = fin.pivot_table(index="quarter", columns="group", values="total_amount", aggfunc="sum").fillna(0)
pct = by_q.div(quarterly_spend, axis=0) * 100
pct["all leakage (A1)"] = pct.sum(axis=1)
pct["online only"] = fin[fin["top_counterpart"].isin(M.LEAKAGE_ONLINE)].groupby("quarter")["total_amount"].sum() / quarterly_spend * 100

ax = plots.leakage_lines(pct, ["revolut", "traditional banks", "crypto exchanges", "twint"], "card spend going to other financial providers, per quarter", ylim=5)
plots.save_fig(ax.figure, "04_leakage_by_destination", "leakage")
pct.round(2)

In [ ]:
ax = plots.leakage_lines(pct, ["all leakage (A1)", "online only"], "leakage, both readings", ylim=20)
plots.save_fig(ax.figure, "05_leakage_two_readings", "leakage")
first, last = pct.iloc[0], pct.iloc[-1]
for col in ["revolut", "traditional banks", "crypto exchanges", "all leakage (A1)", "online only"]:
    print(f"{col:20s} {first[col]:5.2f} % → {last[col]:5.2f} %")

Revolut 3.0 → 4.2 % of all card spend. crypto peaks with bitcoin in 2021 (2.6–2.7 %) and fades to under 1 %. the bank rows grow from 0.9 to 2.6 %, whatever they are. online-only leakage sits at 4.4–6.6 % of spend the whole time, all leakage under A1 at 11–17 % (the `other` rows of `cash` included).

## how many customers, how often, how much

`n_customers` per counterpart and month against monthly active customers. average transfer per customer and per transaction.

In [ ]:
rev = fin[fin["top_counterpart"] == "revolut"].groupby("date").agg(customers=("n_customers", "sum"), amount=("total_amount", "sum"), tx=("n_transactions", "sum"))
rev["pct_active"] = rev["customers"] / monthly["active_customers"] * 100
rev["chf_per_customer"] = rev["amount"] / rev["customers"]
rev["chf_per_tx"] = rev["amount"] / rev["tx"]

ax = plots.monthly_bar(rev["pct_active"], "customers topping up Revolut, % of monthly active", "%")
plots.save_fig(ax.figure, "06_revolut_users", "leakage")
print(f"share of active customers with a Revolut top-up: mean {rev['pct_active'].mean():.1f} %, range {rev['pct_active'].min():.1f}–{rev['pct_active'].max():.1f} %")
print(f"per topping-up customer and month: median {rev['chf_per_customer'].median():.0f} CHF in {rev['tx'].sum() / rev['customers'].sum():.1f} transactions")

In [ ]:
dest = {"revolut": "revolut", "binance": "crypto exchanges", "coinbase": "crypto exchanges", "postfinance": "traditional banks",
        "ubs": "traditional banks", "kantonalbank": "traditional banks", "twint": "twint"}
per = fin.assign(group=fin["top_counterpart"].map(dest)).dropna(subset=["group"]).groupby(["quarter", "group"]).agg(amount=("total_amount", "sum"), tx=("n_transactions", "sum"), customers=("n_customers", "sum"))
per["chf_per_tx"] = per["amount"] / per["tx"]
size = per["chf_per_tx"].unstack()

fig, ax = plt.subplots(figsize=(8, 3.6))
x = size.index.to_timestamp()
for i, col in enumerate(["revolut", "traditional banks", "crypto exchanges", "twint"]):
    style.label_end(ax, x, size[col], col, style.C[i])
style.clean(ax, "average transfer per transaction, CHF", "CHF")
plots.save_fig(fig, "07_transfer_size", "leakage")
size.round(0)

## more customers or more leakage per customer

Revolut amount per month = active customers × share of them topping up × CHF per topping-up customer. all three indexed to their 2021 average.

In [ ]:
dec = pd.DataFrame({
    "active customers": monthly["active_customers"],
    "share topping up": rev["pct_active"],
    "CHF per topping-up customer": rev["chf_per_customer"],
    "Revolut amount": rev["amount"],
})
base = dec[dec.index.year == 2021].mean()
idx = dec / base * 100
q = idx.resample("QS").mean()

fig, ax = plt.subplots(figsize=(8, 3.8))
for i, col in enumerate(dec.columns):
    ax.plot(q.index, q[col], color=style.C[i] if col != "Revolut amount" else style.INK, lw=2, label=col)
ax.axhline(100, color=style.GREY, lw=1)
plots.month_axis(ax)
style.clean(ax, "Revolut top-ups decomposed, index 2021 = 100, quarterly means", "index")
ax.legend(loc="upper left", fontsize=8)
plots.save_fig(fig, "08_decomposition", "leakage")
print("2023 vs 2021 average:", ((dec[dec.index.year == 2023].mean() / base - 1) * 100).round(0).astype(int).to_dict())

the Revolut amount grows +130 % from 2021 to 2023. the customer base explains +50 %, each topping-up customer sending more per month another +57 %. the share of customers who top up does not move (−3 %). so it is not spreading to more customers, the ones who do it are doing it harder.

## the customer side

leakage share per customer, how many have any, and where they sit in tenure and spend.

In [ ]:
has = feats["leak_share"] > 0
print(f"customers with any cash/savings activity: {has.mean():.1%}, with more than 5 % of spend: {(feats['leak_share'] > 0.05).mean():.1%}, more than 50 %: {(feats['leak_share'] > 0.5).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(feats.loc[has, "leak_share"], bins=40, color=style.C[0])
style.clean(axes[0], "leakage share, customers with any", "customers", "share of spend")
by_t = feats.groupby("tenure_bucket", observed=True)["leak_share"].mean() * 100
axes[1].bar(by_t.index.astype(str), by_t.values, color=style.C[0], width=0.7)
style.clean(axes[1], "mean leakage share by tenure", "%")
fig.tight_layout()
plots.save_fig(fig, "09_customer_leakage", "leakage")

feats["spend_decile"] = pd.qcut(feats["total_amount"], 10, labels=range(1, 11))
feats.groupby("spend_decile", observed=True).agg(leak_share=("leak_share", "mean"), any_leak=("has_leak", "mean"), leak_chf=("leak_amount", "sum")).round(3)

## a look ahead

Revolut share of spend, monthly, with a damped-trend exponential smoothing fit and a seasonal-naive baseline. backtest on the last six months. 36 points, so this is a direction, not a number.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

rev_share = (rev["amount"] / monthly["spend"] * 100).asfreq("MS")
train_s, test_s = rev_share.iloc[:-6], rev_share.iloc[-6:]

ets = ExponentialSmoothing(train_s, trend="add", damped_trend=True, seasonal=None).fit()
naive = train_s.iloc[-12:-6].to_numpy()
mae_ets = np.mean(np.abs(ets.forecast(6).to_numpy() - test_s.to_numpy()))
mae_naive = np.mean(np.abs(naive - test_s.to_numpy()))
print(f"backtest MAE, last 6 months: ETS {mae_ets:.2f} pts, seasonal naive {mae_naive:.2f} pts")

full = ExponentialSmoothing(rev_share, trend="add", damped_trend=True, seasonal=None).fit()
fc = full.forecast(12)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(rev_share.index, rev_share.values, color=style.C[0], lw=2, label="Revolut, % of spend")
ax.plot(fc.index, fc.values, color=style.C[1], lw=2, ls="--", label="ETS forecast 2024")
ax.plot(rev_share.index, full.fittedvalues, color=style.GREY, lw=1)
plots.month_axis(ax)
style.clean(ax, "Revolut share of card spend and a 12-month forecast", "%")
ax.legend(loc="upper left")
plots.save_fig(fig, "10_revolut_forecast", "leakage")
print(f"forecast Dec 2024: {fc.iloc[-1]:.1f} %, last actual {rev_share.iloc[-1]:.1f} %")

## numbers for the slides

- Revolut: 3.0 % of card spend in 2021Q1, 4.2 % in 2023Q4
- one in twenty active customers tops up Revolut in any given month, steady over three years
- Revolut amount +130 % 2021 → 2023: +50 % customer base, +57 % per topping-up customer, the share of customers topping up unchanged
- crypto exchanges peak with bitcoin in 2021 and fade to under 1 % of spend by 2023
- online-only leakage (card top-ups of Revolut, crypto, Twint) 4–7 % of spend, all cash + savings 11–17 % (A1)
- a damped-trend forecast continues the slow rise to about 4.4 % by end of 2024. it is no better than a seasonal naive guess on the backtest, so it is a direction, not a number

every leakage number on a slide carries "assumes cash and savings are transfers, see A1", or uses the online-only reading.